# 0. Setup

In [1]:
%pip install -qU openai langchain langchain-openai langchain-chroma datasets tiktoken chromadb python-dotenv --upgrade

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 2.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.4/948.4 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.6/503.6 kB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.8/19.8 MB 76.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.2/284.2 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 70.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.5/447.5 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 82.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.5 MB/s et

In [2]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain.schema import Document
from langchain.chains import RetrievalQA
from google.colab import userdata

api_key = userdata.get('OPENAI_API_KEY')

# Initialize LangChain OpenAI client and embeddings
llm = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=api_key,
    temperature=0
)


# 1. Extracting files

In [3]:
from google.colab import files

In [4]:
uploaded = files.upload()

Saving RenesasRH850F1K_ADConverter.pdf to RenesasRH850F1K_ADConverter.pdf
Saving RenesasRH850F1K_ResetFactorRegister.pdf to RenesasRH850F1K_ResetFactorRegister.pdf


In [5]:
for filename in uploaded.keys():
    print(f'Subido: {filename}')

Subido: RenesasRH850F1K_ADConverter.pdf
Subido: RenesasRH850F1K_ResetFactorRegister.pdf


In [6]:
!pip install PyMuPDF

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 23.4 MB/s eta 0:00:00


In [7]:
import fitz  # PyMuPDF

# Función para extraer texto de un PDF
def extract_text_from_pdf(pdf_path):
    text = ""
    with fitz.open(pdf_path) as pdf:
        for page in pdf:
            text += page.get_text()
    return text

# Extraer texto de los archivos PDF subidos
pdf_texts = {}
for filename in uploaded.keys():
    pdf_texts[filename] = extract_text_from_pdf(filename)

# Mostrar el texto extraído de los PDFs
for filename, text in pdf_texts.items():
    print(f'Texto extraído de {filename}:\n{text[:500]}...')  # Muestra los primeros 500 caracteres

Texto extraído de RenesasRH850F1K_ADConverter.pdf:
R01UH0562EJ0110 Rev.1.10
Page 2569 of 2863
Nov 30, 2016
RH850/F1K
Section 31 A/D Converter (ADCA)
Section 31
A/D Converter (ADCA)
This section contains a generic description of the A/D Converter (ADCA).
The first part of this section describes all RH850/F1K specific properties, such as the number of units, 
register base addresses, etc. The remainder of the section describes the functions and registers of the 
ADCA.
31.1
Features of RH850/F1K ADCA
31.1.1
Number of Units and Channels
This microco...
Texto extraído de RenesasRH850F1K_ResetFactorRegister.pdf:
R01UH0562EJ0110 Rev.1.10
Page 406 of 2863
Nov 30, 2016
RH850/F1K
Section 9 Reset Controller
9.3.2
 Details of Reset Flag Registers
9.3.2.1
RESF ² Reset Factor Register
This register contains information about which type of resets occurred after the last power-on clear 
reset. This register is initialized by a power-up reset PURES.
Each reset condition sets the corresponding flag in t

# 2. Text Splitting into Chunks

In [8]:
def split_text_into_chunks(text, max_chunk_size=500):
    # Dividir el texto en párrafos
    paragraphs = text.split('\n')
    chunks = []

    current_chunk = ""
    for paragraph in paragraphs:
        # Si el párrafo actual más el nuevo párrafo excede el tamaño máximo, guardar el chunk actual
        if len(current_chunk) + len(paragraph) + 1 > max_chunk_size:
            chunks.append(current_chunk.strip())
            current_chunk = paragraph  # Comenzar un nuevo chunk
        else:
            current_chunk += " " + paragraph  # Agregar el párrafo al chunk actual

    # Agregar el último chunk si no está vacío
    if current_chunk:
        chunks.append(current_chunk.strip())

    return chunks

In [9]:
# Crear un diccionario para almacenar los fragmentos
pdf_chunks = {}

# Dividir el texto de cada PDF en fragmentos
for filename, text in pdf_texts.items():
    pdf_chunks[filename] = split_text_into_chunks(text, max_chunk_size=500)

# Mostrar algunos fragmentos de cada PDF
for filename, chunks in pdf_chunks.items():
    print(f'Fragmentos de {filename}:')
    for i, chunk in enumerate(chunks[:3]):  # Mostrar solo los primeros 3 fragmentos
        print(f'Chunk {i + 1}: {chunk}\n')

Fragmentos de RenesasRH850F1K_ADConverter.pdf:
Chunk 1: R01UH0562EJ0110 Rev.1.10 Page 2569 of 2863 Nov 30, 2016 RH850/F1K Section 31 A/D Converter (ADCA) Section 31 A/D Converter (ADCA) This section contains a generic description of the A/D Converter (ADCA). The first part of this section describes all RH850/F1K specific properties, such as the number of units,  register base addresses, etc. The remainder of the section describes the functions and registers of the  ADCA. 31.1 Features of RH850/F1K ADCA 31.1.1 Number of Units and Channels

Chunk 2: This microcontroller has the following number of ADCA units. An ADCAn unit has the same number of physical channels as the number of A/D input pins and the  same number of virtual channels as the number of addresses where the results of A/D conversion will  be stored. The numbers of channels on individual products are as listed below. Note 1. When 10-bit mode is selected, this pin can be used for 10-bit conversion. Note 2.

Chunk 3: When 12-b

# 3. Embedding

In [10]:
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=api_key
)

# Crear un diccionario para almacenar los embeddings
pdf_embeddings = {}

# Iterar sobre los fragmentos y crear embeddings
for filename, chunks in pdf_chunks.items():
    pdf_embeddings[filename] = []
    for chunk in chunks:
        embedding = embeddings.embed_documents(chunk)  # Crear el embedding para el chunk individualmente
        pdf_embeddings[filename].append(embedding)

# Mostrar algunos embeddings generados
for filename, embeddings in pdf_embeddings.items():
    print(f'Embeddings de {filename}:')
    for i, embedding in enumerate(embeddings[:3]):  # Mostrar solo los primeros 3 embeddings
        print(f'Embedding {i + 1}: {embedding[:5]}...')  # Muestra los primeros 5 valores del embedding


Embeddings de RenesasRH850F1K_ADConverter.pdf:
Embedding 1: [[0.003252720460295677, 0.04339371249079704, -0.018326612189412117, 0.008613652549684048, 0.008512400090694427, -0.018558045849204063, -0.012605873867869377, -0.016142461448907852, 0.00478777289390564, -0.05386606231331825, -0.013169991783797741, -0.008779995143413544, -0.035756420344114304, -0.007760242559015751, 0.03251635655760765, -0.011622282676398754, 0.03902541473507881, 0.003001398639753461, -0.020539691671729088, 0.05895759165287018, -0.002840480301529169, -0.024922456592321396, 0.01900644600391388, -0.042959775775671005, 0.03364459425210953, 0.0176178477704525, -0.02712107077240944, -0.006176372058689594, 0.018008390441536903, -0.017024800181388855, -0.006512673106044531, -0.0354960598051548, -0.02171132154762745, -0.04099259525537491, -0.032545287162065506, -0.02968130074441433, 0.003062872914597392, -0.001636304659768939, -0.01369794923812151, 0.06578487157821655, 0.002462593372911215, -0.032979223877191544, -0.007

# 4. Vector Stores

In [11]:
from langchain.schema import Document
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
import os

# Inicializar embeddings
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=api_key
)

# Inicializar Chroma
persist_directory = "./chroma_db"
vectorstore = Chroma(
    collection_name="RH850_F1K",
    embedding_function=embeddings,
    persist_directory=persist_directory
)

# Guardar los documentos en Chroma
for filename, chunks in pdf_chunks.items():
    documents = [
        Document(page_content=chunk, metadata={"filename": filename})
        for chunk in chunks
    ]
    vectorstore.add_documents(documents)

# 5. Retriving from the Persistant Vector Datastore

In [16]:

# Definir la consulta
query = "¿Cuál es el nombre del bit (Bitname) del WDTA0 reset flag?"

# Recuperar documentos relevantes
results = vectorstore.similarity_search(query, k=3)  # k es el número de documentos que deseas recuperar

# Mostrar los resultados
for i, result in enumerate(results):
    print(f"Resultado {i + 1}:")
    print(f"Contenido: {result.page_content}")
    print(f"Metadata: {result.metadata}\n")

Resultado 1:
Contenido: RESF1 WDTA0 reset flag 0: No reset occurred 1: Reset has occurred 0 RESF0 Software reset flag 0: No reset occurred 1: Reset has occurred Table 9.5 RESF Register Contents  (2/2) Bit Position Bit Name Function
Metadata: {'filename': 'RenesasRH850F1K_ResetFactorRegister.pdf'}

Resultado 2:
Contenido: 7 RESF7 CVM reset flag 0: No reset occurred 1: Reset has occurred 6 RESF6 LVI reset flag 0: No reset occurred 1: Reset has occurred 5 RESF5 CLMA2 reset flag 0: No reset occurred 1: Reset has occurred 4 RESF4 CLMA1 reset flag 0: No reset occurred 1: Reset has occurred 3 RESF3 CLMA0 reset flag 0: No reset occurred 1: Reset has occurred R01UH0562EJ0110 Rev.1.10 Page 407 of 2863 Nov 30, 2016 RH850/F1K Section 9 Reset Controller 2 RESF2 WDTA1 reset flag 0: No reset occurred 1: Reset has occurred 1
Metadata: {'filename': 'RenesasRH850F1K_ResetFactorRegister.pdf'}

Resultado 3:
Contenido: R01UH0562EJ0110 Rev.1.10 Page 406 of 2863 Nov 30, 2016 RH850/F1K Section 9 Reset Control

#6. Retrivers in Langchain

In [17]:
from langchain.chains import RetrievalQA

# Configurar el retriever
retriever = vectorstore.as_retriever()

In [18]:
# Crear la cadena de pregunta y respuesta
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,  # Tu modelo de ChatOpenAI
    chain_type="stuff",  # Tipo de cadena que deseas usar
    retriever=retriever,
    return_source_documents=True  # Para devolver los documentos fuente si es necesario
)

In [19]:
# Definir la consulta
query = "¿Cuál es el nombre del bit (Bitname) del WDTA0 reset flag?"

# Obtener la respuesta
response = qa_chain({"query": query})

# Mostrar la respuesta generada
print("Respuesta:", response['result'])
print("Documentos fuente:", response['source_documents'])

Respuesta: El nombre del bit del WDTA0 reset flag es RESF1.
Documentos fuente: [Document(id='09c81ea2-fbe8-43ac-8ea7-6c5bb51688e0', metadata={'filename': 'RenesasRH850F1K_ResetFactorRegister.pdf'}, page_content='RESF1 WDTA0 reset flag 0: No reset occurred 1: Reset has occurred 0 RESF0 Software reset flag 0: No reset occurred 1: Reset has occurred Table 9.5 RESF Register Contents  (2/2) Bit Position Bit Name Function'), Document(id='abdae905-9e30-4974-9bc6-ec139490773b', metadata={'filename': 'RenesasRH850F1K_ResetFactorRegister.pdf'}, page_content='7 RESF7 CVM reset flag 0: No reset occurred 1: Reset has occurred 6 RESF6 LVI reset flag 0: No reset occurred 1: Reset has occurred 5 RESF5 CLMA2 reset flag 0: No reset occurred 1: Reset has occurred 4 RESF4 CLMA1 reset flag 0: No reset occurred 1: Reset has occurred 3 RESF3 CLMA0 reset flag 0: No reset occurred 1: Reset has occurred R01UH0562EJ0110 Rev.1.10 Page 407 of 2863 Nov 30, 2016 RH850/F1K Section 9 Reset Controller 2 RESF2 WDTA1 res

In [20]:
# Definir la consulta
query = "¿Qué información tiene el registro RESF0?"

# Obtener la respuesta
response = qa_chain({"query": query})

# Mostrar la respuesta generada
print("Respuesta:", response['result'])
print("Documentos fuente:", response['source_documents'])

Respuesta: El registro RESF0 contiene la bandera de reinicio por software. Su función es indicar si ha ocurrido un reinicio por software, donde un valor de 0 significa que no ha ocurrido ningún reinicio y un valor de 1 significa que ha ocurrido un reinicio.
Documentos fuente: [Document(id='d4ff5d22-157c-4457-ad9b-16071c84dd7d', metadata={'filename': 'RenesasRH850F1K_ResetFactorRegister.pdf'}, page_content='RESF reads 0000 000AH. Access: This register is a read-only register that can be read in 32-bit units. Address: FFF8 0760H Value after reset: 0000 0200H / 0000 0300H Bit 31 30 29 28 27 26 25 24 23 22 21 20 19 18 17 16 ² ² ² ² ² ² ² ² ² ² ² ² ² ² ² ² Value after reset 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 R/W R R R R R R R R R R R R R R R R Bit 15 14 13 12 11 10 9 8 7 6 5 4 3 2 1 0 ² ² ² ² ² RESF 10 RESF9 RESF8 RESF7 RESF6 RESF5 RESF4 RESF3 RESF2 RESF1 RESF0 Value after reset 0 0 0 0 0 0 1 1/0*1 0 0 0'), Document(id='09c81ea2-fbe8-43ac-8ea7-6c5bb51688e0', metadata={'filename': 'RenesasRH850

In [21]:
# Definir la consulta
query = "¿Cuál es la resolución del ADCA?"

# Obtener la respuesta
response = qa_chain({"query": query})

# Mostrar la respuesta generada
print("Respuesta:", response['result'])
print("Documentos fuente:", response['source_documents'])

Respuesta: El ADCA tiene una resolución de 10 bits o 12 bits, dependiendo de la configuración seleccionada.
Documentos fuente: [Document(id='fddb6211-8600-4744-82c2-e8763e6b3360', metadata={'filename': 'RenesasRH850F1K_ADConverter.pdf'}, page_content='This microcontroller has the following number of ADCA units. An ADCAn unit has the same number of physical channels as the number of A/D input pins and the  same number of virtual channels as the number of addresses where the results of A/D conversion will  be stored. The numbers of channels on individual products are as listed below. Note 1. When 10-bit mode is selected, this pin can be used for 10-bit conversion. Note 2.'), Document(id='2b22e4ef-a0a5-4339-827c-b1551e93b3f6', metadata={'filename': 'RenesasRH850F1K_ADConverter.pdf'}, page_content='ADCA1TRG0 External trigger pin (scan group 1)*1 ADCA1TRG0 ADCA1TRG1 External trigger pin (scan group 2)*1 ADCA1TRG1 ADCA1TRG2 External trigger pin (scan group 3)*1 ADCA1TRG2 R01UH0562EJ0110 Rev.

In [23]:
# Definir la consulta
query = "¿Cuántos pines tiene disponibles el ADCA para una resolución de 12 bits?"

# Obtener la respuesta
response = qa_chain({"query": query})

# Mostrar la respuesta generada
print("Respuesta:", response['result'])
print("Documentos fuente:", response['source_documents'])

Respuesta: El ADCA tiene disponibles 16 pines para conversión en modo de 12 bits.
Documentos fuente: [Document(id='f477045d-df3c-4965-8210-90eebd1416a6', metadata={'filename': 'RenesasRH850F1K_ADConverter.pdf'}, page_content='16 16 10 bit pin for  conversion*2 20 20 20 ADCA1 12 bit pin for  conversion*1 ² 8 16 10 bit pin for  conversion*2 ² 4 8 Table 31.3 Unit Configurations and Virtual Channels Unit Name (Number of Channels) ADCAn RH850/F1K 100 pins RH850/F1K 144 pins RH850/F1K 176 pins ADCA0 50 50 50 ADCA1 ² 36 36 R01UH0562EJ0110 Rev.1.10 Page 2570 of 2863 Nov 30, 2016 RH850/F1K Section 31 A/D Converter (ADCA) The following table shows values indicated by the indices of each product. 31.1.2 Register Base Address'), Document(id='fddb6211-8600-4744-82c2-e8763e6b3360', metadata={'filename': 'RenesasRH850F1K_ADConverter.pdf'}, page_content='This microcontroller has the following number of ADCA units. An ADCAn unit has the same number of physical channels as the number of A/D input pins a

In [24]:
# Definir la consulta
query = "¿Cuál es la dirección base para el registro ADCA0_base?"

# Obtener la respuesta
response = qa_chain({"query": query})

# Mostrar la respuesta generada
print("Respuesta:", response['result'])
print("Documentos fuente:", response['source_documents'])

Respuesta: La dirección base para el registro ADCA0_base es FFF2 0000H.
Documentos fuente: [Document(id='ab1a6d5c-e3f2-449c-8d3d-0171dc1581a9', metadata={'filename': 'RenesasRH850F1K_ADConverter.pdf'}, page_content='ADCAn base addresses are listed in the following table. ADCAn register addresses are given as offsets from the base addresses. Table 31.4 Indices Index Description n Throughout this section, the individual ADCA units are identified by the index ³n´ (n = 0, 1); for  example, ADCAnPWDVCR indicates the PWM-Diag virtual channel register. m Throughout this section, the individual physical channels (channels in the unit) of ADCAn are  identified by the index ³m´; for example, ANInm. j'), Document(id='475c3ab5-b425-4d68-b7b0-60fc6832b3a0', metadata={'filename': 'RenesasRH850F1K_ADConverter.pdf'}, page_content='x = 1 to 3 (ADCA0) x = 1 to 3 (ADCA1) k = 0 to 5 (ADCA0) k = 0 to 5 (ADCA0) k = 0 to 5 (ADCA0) Table 31.6 Register Base Addresses Base Address Name Base Address <ADCA0_base>

In [25]:
# Definir la consulta
query = "¿Cuál es la resolución en bits de la señal de unidad ANI108 en el ADCA?"

# Obtener la respuesta
response = qa_chain({"query": query})

# Mostrar la respuesta generada
print("Respuesta:", response['result'])
print("Documentos fuente:", response['source_documents'])

Respuesta: No sé.
Documentos fuente: [Document(id='4b0ca47c-d5d7-47e3-9c8a-576d95de84a9', metadata={'filename': 'RenesasRH850F1K_ADConverter.pdf'}, page_content='ADCA1I8 12 ² ² ² ANI109 ADCA1I9 12 ² ² ² ANI110 ADCA1I10 12 ² ² ² ANI111 ADCA1I11 12 ² ² ² ANI112 ADCA1I12 12 ² ² ² ANI113 ADCA1I13 12 ² ² ² ANI114 ADCA1I14 12 ² ² ² ANI115 ADCA1I15 12 ² ² ² ANI116 ADCA1I0S 10 ² ² ANI117 ADCA1I1S 10 ² ² ANI118 ADCA1I2S 10 ² ² ANI119 ADCA1I3S 10 ² ² ANI120 ADCA1I4S 10 ² ² ² ANI121 ADCA1I5S 10 ² ² ² ANI122 ADCA1I6S 10 ² ² ² ANI123 ADCA1I7S 10 ² ² ² Table 31.13 ADCA1 External Input/Output Signals  Unit Signal Name Description Alternative Port Pin  Signal ADCA1'), Document(id='ee088ee3-9c9a-451a-9304-01510ed62fb7', metadata={'filename': 'RenesasRH850F1K_ADConverter.pdf'}, page_content='ANI022 ADCA0I6S 10 ² ANI023 ADCA0I7S 10 ² ANI024 ADCA0I8S 10 ² ANI025 ADCA0I9S 10 ² ANI026 ADCA0I10S 10 ² ANI027 ADCA0I11S 10 ² ANI028 ADCA0I12S 10 ² ANI029 ADCA0I13S 10 ² ANI030 ADCA0I14S 10 ² ANI031 ADCA0I15S 10 ²